# 08 · Rete di co-occorrenza dei tag del Guardian

Questo notebook costruisce per il Guardian la stessa rete di co-occorrenza costruita per il NYT nel notebook 02, così che le due reti si possano confrontare e che la community detection del notebook 10 possa lavorare su entrambe. I **nodi** sono i tag tematici che la redazione del Guardian assegna agli articoli (tipo `keyword`); un **arco** collega due tag che compaiono nello stesso articolo, con **peso** pari al numero di articoli in cui compaiono insieme.

**Input.** `articles_ice_relevant.parquet`, solo gli articoli `is_ice_relevant` (filtro di rilevanza del notebook 05, §5). Il Parquet contiene solo gli identificativi dei tag; nome leggibile e tipo si rileggono dai file raw (`data/raw/guardian/search/**/*.json`), senza nuove chiamate all'API.

**Cosa è identico al NYT.** Il codice di costruzione e diluizione (`build_cooccurrence_graph`, `prune_graph`, `threshold_table`), i quattro periodi, il grafo dinamico, la soglia del grafo diluito (`MIN_EDGE_WEIGHT = 5`) e gli attributi dei nodi (`count`, `avg_rank`, `category`).

**Cosa cambia per forza**, perché le due testate hanno tassonomie diverse:

| | NYT | Guardian |
|---|---|---|
| Keyword | `keywords[].value` | tag `keyword` (id `sezione/slug`), etichetta = `webTitle` |
| Nodo ICE escluso | `immigration and customs enforcement (us)` | `us-news/ice-us-immigration-and-customs-enforcement` |
| Categoria | `keywords[].name` | `keywordType` del tag, ricondotto alle categorie del NYT |
| Rank | `keywords[].rank` | posizione del tag nella lista dell'articolo (1 = primo) |

**Struttura.** §0 setup → §1 articoli e tag → §2 grafo del corpus intero → §3 grafo diluito → §4 grafi per periodo e grafo dinamico → §5 confronto di struttura NYT vs Guardian → §6 lettura dei risultati e limiti.

**Output** in `data/processed/guardian/`: `ice_topic_network.gexf` (intero), `ice_topic_network_pruned.gexf` (diluito), `ice_topic_network_{periodo}.gexf`, `ice_topic_network_dynamic.gexf`, `topic_network_period_stats.csv`, `topic_network_nyt_vs_guardian.csv`, `plots/edge_weight_hist.png`. Figure per la tesi in `docs/figures/`.

## 0. Setup

Import, percorsi e costanti. Il notebook legge solo dati già su disco: gli articoli rilevanti (notebook 05), i file raw del Guardian per i metadati dei tag e, nel §5, i grafi NYT scritti dal notebook 02, che va quindi eseguito prima.

- `ICE_TAG`: il tag dell'agenzia, escluso dai nodi (§1);
- `MIN_EDGE_WEIGHT = 5`: soglia del grafo diluito, la stessa del NYT, per confrontare le due reti a parità di regola (§3).

In [ ]:
import json
import shutil
import logging
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display

%matplotlib inline

# Radice del progetto: funziona sia lanciando da notebooks/ sia dalla radice
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Raw Guardian: serve solo per leggere nome e tipo dei tag
RAW_DIR = BASE_DIR / "data" / "raw" / "guardian" / "search"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "guardian"
PLOTS_DIR = PROCESSED_DIR / "plots"
# Grafi NYT del notebook 02, usati solo nel confronto del §5
NYT_DIR = BASE_DIR / "data" / "processed" / "nyt"

# Input e output: articoli rilevanti, GEXF del NYT, GEXF e CSV del Guardian
IN_RELEVANT = PROCESSED_DIR / "articles_ice_relevant.parquet"
IN_NYT_GEXF = NYT_DIR / "ice_topic_network.gexf"
IN_NYT_PRUNED_GEXF = NYT_DIR / "ice_topic_network_pruned.gexf"
OUT_GEXF = PROCESSED_DIR / "ice_topic_network.gexf"
OUT_PRUNED_GEXF = PROCESSED_DIR / "ice_topic_network_pruned.gexf"
OUT_DYNAMIC_GEXF = PROCESSED_DIR / "ice_topic_network_dynamic.gexf"
OUT_PERIOD_STATS = PROCESSED_DIR / "topic_network_period_stats.csv"
OUT_COMPARISON = PROCESSED_DIR / "topic_network_nyt_vs_guardian.csv"
OUT_WEIGHT_HIST_PNG = PLOTS_DIR / "edge_weight_hist.png"

# Tag dell'agenzia: è il criterio di selezione del corpus, quindi non è un nodo informativo
ICE_TAG = "us-news/ice-us-immigration-and-customs-enforcement"
# stessa soglia del grafo diluito NYT (notebook 02)
MIN_EDGE_WEIGHT = 5

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("guardian_topic_network")

FIG_DIR = BASE_DIR / "docs" / "figures"  # figure per la tesi
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"BASE_DIR: {BASE_DIR}")

## 1. Caricamento degli articoli e dei tag

Il Parquet del notebook 05 conserva per ogni articolo solo gli identificativi dei tag (`keyword_tags_json`). Il nome leggibile (`webTitle`) e il tipo (`keywordType`) stanno nelle risposte dell'API salvate nel raw, e si rileggono da lì.

**Categorie.** Il `keywordType` del Guardian è ricondotto alle categorie del NYT, così le due reti usano la stessa colorazione in Gephi: `person` → `persons`, `place` → `glocations`, `organisation` → `organizations`, `work-of-art-or-product` → `creative_works`. `event` e `other` finiscono in `subject`, perché il NYT non ha una categoria per gli eventi e li tratta come temi.

**Il tag ICE è escluso** dai nodi, come la keyword ICE nel NYT: collegherebbe quasi tutti gli altri nodi senza dire nulla sui temi. Le due esclusioni non sono però equivalenti. Nel NYT la keyword ICE è presente in tutti gli articoli per costruzione, perché è il criterio di selezione del corpus; nel Guardian il tag è presente in 964 articoli su 1.738, quasi tutti del 2025-2026, perché la redazione lo applica con regolarità solo da allora (notebook 05).

**Rank.** Il NYT assegna a ogni keyword un `rank` di importanza; il Guardian no. Al suo posto si usa la posizione del tag nella lista dell'articolo (1 = primo). È un sostituto, non la stessa misura: il `avg_rank` dei due grafi non va confrontato direttamente.

In [ ]:
# keywordType del Guardian -> categoria NYT equivalente (event e other diventano subject)
CATEGORY_MAP = {
    "other": "subject",
    "event": "subject",
    "person": "persons",
    "place": "glocations",
    "organisation": "organizations",
    "work-of-art-or-product": "creative_works",
}


def load_tag_metadata() -> dict[str, dict]:
    """Legge dai file raw nome e categoria di ogni tag: {tag_id: {"label": webTitle, "category": ...}}.

    Scorre tutte le pagine di risultati salvate (entrambe le query, tutti gli anni) e, per ogni tag
    incontrato, registra l'etichetta leggibile e la categoria NYT equivalente (CATEGORY_MAP).
    Copre anche i tag degli articoli non rilevanti: è un dizionario di consultazione, non un filtro.
    """
    tags = {}
    for path in sorted(RAW_DIR.glob("*/*.json")):
        for page in json.loads(path.read_text()):
            for result in page["results"]:
                for tag in result.get("tags") or []:
                    tags[tag["id"]] = {
                        "label": tag["webTitle"],
                        "category": CATEGORY_MAP.get(tag.get("keywordType"), "subject"),
                    }
    return tags


def load_ice_articles() -> pd.DataFrame:
    """Carica gli articoli rilevanti con id, data e lista dei tag (decodificata dal JSON)."""
    df = pd.read_parquet(IN_RELEVANT, columns=["id", "pub_date", "keyword_tags_json", "is_ice_relevant"])
    df = df[df["is_ice_relevant"]].copy()
    df["keywords"] = df["keyword_tags_json"].apply(json.loads)
    return df[["id", "pub_date", "keywords"]]


def extract_keywords(keywords: list) -> list[tuple[str, int, str]]:
    """Tag di un singolo articolo come lista di (tag_id, rank, categoria), senza il tag ICE.

    Il rank è la posizione del tag nella lista originale (1 = primo), calcolata prima di togliere
    il tag ICE; i tag ripetuti nello stesso articolo vengono contati una volta sola.
    """
    keywords_filtered = []
    seen = set()
    for rank, tag_id in enumerate(keywords, start=1):
        if tag_id == ICE_TAG or tag_id in seen:
            continue
        seen.add(tag_id)
        keywords_filtered.append((tag_id, rank, TAG_META.get(tag_id, {}).get("category", "subject")))
    return keywords_filtered


TAG_META = load_tag_metadata()
df = load_ice_articles()
# Tag per articolo dopo l'esclusione di ICE, per descrivere il corpus
n_tags = df["keywords"].apply(lambda tags: len(extract_keywords(tags)))
log.info("Articoli Guardian rilevanti: %d, tag noti dal raw: %d", len(df), len(TAG_META))
log.info("Tag per articolo (senza ICE): mediana %d, media %.1f, articoli senza tag %d", n_tags.median(), n_tags.mean(), (n_tags == 0).sum())

Il log riporta 1.738 articoli rilevanti e 1.263 tag diversi letti dal raw (su tutti i candidati, non solo sui rilevanti). Escluso ICE, un articolo ha in mediana 6 tag (media 6,1); 4 articoli non ne hanno nessuno e diventano nodi isolati o non contribuiscono al grafo. Il NYT assegna più keyword per articolo (mediana 8): a parità di articoli, la rete del NYT ha più nodi e più archi.

In [ ]:
# Demo: tag estratti dal primo articolo
sample_row = df.iloc[0]
print(f"Articolo: {sample_row['id']}")
extract_keywords(sample_row["keywords"])

L'esempio mostra il formato prodotto da `extract_keywords` sul primo articolo del corpus (2007): due tag, entrambi di sezione (`world/world`, `us-news/us-news`), con la loro posizione e la categoria. È un caso tipico degli articoli più vecchi, che hanno pochi tag e spesso solo quelli di sezione; il §2 mostra quanto questi tag pesano nella rete.

## 2. Grafo di co-occorrenza (corpus intero)

La funzione è la stessa del notebook 02. Per ogni articolo:

- ogni tag diventa un nodo, o incrementa il contatore `count` (numero di articoli in cui compare) se esiste già;
- ogni coppia di tag dello stesso articolo diventa un arco, o ne incrementa il `weight` (numero di articoli in cui i due tag compaiono insieme).

Ogni articolo con *k* tag aggiunge quindi una clique di *k*(*k*−1)/2 archi: con 6 tag, 15 coppie. Alla fine ogni nodo riceve `avg_rank` (posizione media del tag), `category` (la categoria più frequente del tag) e `label` (nome leggibile), che Gephi usa come etichetta al posto dell'id. Il grafo è scritto in formato GEXF.

In [ ]:
def build_cooccurrence_graph(df: pd.DataFrame) -> nx.Graph:
    """Costruisce il grafo di co-occorrenza dei tag degli articoli in df.

    Per ogni articolo: aggiunge i suoi tag come nodi (o ne incrementa count e la somma dei rank) e
    ogni coppia di tag come arco (o ne incrementa weight). Alla fine calcola per ogni nodo avg_rank,
    la categoria più frequente e l'etichetta leggibile. Un articolo con k tag aggiunge k(k-1)/2 coppie.
    """
    graph = nx.Graph()

    for _, row in df.iterrows():
        keywords = extract_keywords(row["keywords"])

        for val, rank, category in keywords:
            if graph.has_node(val):
                graph.nodes[val]["count"] += 1
                graph.nodes[val]["sum_rank"] += rank
                graph.nodes[val]["category_counts"][category] = (
                    graph.nodes[val]["category_counts"].get(category, 0) + 1
                )
            else:
                graph.add_node(val, count=1, sum_rank=rank, category_counts={category: 1})

        if len(keywords) < 2:
            continue

        keywords_values = [kw[0] for kw in keywords]
        edges = combinations(keywords_values, 2)

        for u, v in edges:
            if graph.has_edge(u, v):
                graph[u][v]["weight"] += 1
            else:
                graph.add_edge(u, v, weight=1)

    # Attributi finali dei nodi: rank medio, categoria prevalente, etichetta leggibile
    for node, data in graph.nodes(data=True):
        data["avg_rank"] = data["sum_rank"] / data["count"] if data["count"] > 0 else 0
        category_counts = data.pop("category_counts")
        data["category"] = max(category_counts, key=category_counts.get)
        data["label"] = TAG_META.get(node, {}).get("label", node)

    return graph


def export_graph(graph: nx.Graph, path: Path) -> None:
    """Scrive il grafo in formato GEXF, leggibile da Gephi e dal notebook 10."""
    nx.write_gexf(graph, path)


log.info("Costruisco co-occurrence network su %d articoli (corpus intero)", len(df))
graph = build_cooccurrence_graph(df)
log.info("Grafo intero: %d nodi, %d archi", graph.number_of_nodes(), graph.number_of_edges())
export_graph(graph, OUT_GEXF)
log.info("Scritto %s", OUT_GEXF.relative_to(BASE_DIR))

Il grafo del corpus intero ha 609 nodi e 7.025 archi: è molto più piccolo di quello del NYT (2.441 nodi, 36.654 archi), perché il Guardian usa meno tag diversi e ne assegna meno per articolo.

In [ ]:
# Controllo: i 15 nodi più frequenti devono essere temi sensati, senza rumore
top_nodes = sorted(graph.nodes(data=True), key=lambda x: (-x[1]["count"], x[0]))[:15]
pd.DataFrame(
    [{"tag": n, "label": d["label"], "count": d["count"], "avg_rank": round(d["avg_rank"], 2), "category": d["category"]} for n, d in top_nodes]
)

I 15 tag più frequenti sono tutti pertinenti, ma i primi sono molto generali. `US news` compare in 1.664 articoli su 1.738 (96%) e `World news` in 454: sono tag di sezione, che indicano dove l'articolo è pubblicato più che di cosa parla. Seguono tag ampi (`US immigration` 1.346, `Trump administration` 943, `US politics` 896) e poi luoghi (California, Minnesota, Texas, Minneapolis, Los Angeles) e un tag-evento (`Minnesota ICE shootings`, 111). `US immigration` ha la posizione media più bassa (2,44): quando è presente compare tra i primi tag dell'articolo.

La cella successiva misura quanto pesano nella rete i tag di sezione, che il NYT non ha.

In [ ]:
# Tag di sezione: id della forma "sezione/sezione" (us-news/us-news, world/world, ...)
section_tags = sorted(n for n in graph if n.split("/")[0] == n.split("/")[-1])
total_weight = sum(w for _, _, w in graph.edges(data="weight"))
section_weight = sum(w for u, v, w in graph.edges(data="weight") if u in section_tags or v in section_tags)
print(f"Tag di sezione: {len(section_tags)} su {graph.number_of_nodes()} nodi")
print(f"Quota del peso totale degli archi che tocca un tag di sezione: {100 * section_weight / total_weight:.1f}%")
pd.DataFrame([(n, graph.nodes[n]["count"]) for n in section_tags], columns=["tag", "articoli"]).sort_values("articoli", ascending=False).head(8)

I tag di sezione sono 27 su 609 nodi, ma il 38,8% del peso degli archi coinvolge almeno uno di essi, quasi tutto per `US news` e `World news`. Sono hub di sezione, non temi. Sono stati lasciati nel grafo per non applicare al Guardian un filtro che il NYT non ha; la conseguenza va tenuta presente in ogni misura di centralità sul grafo Guardian. Il notebook 10 usa pesi normalizzati (association strength), che ridimensionano i nodi presenti quasi ovunque.

## 3. Grafo diluito: soglia sui pesi degli archi

Il grafo intero contiene molti archi di peso 1, coppie di tag comparse insieme in un solo articolo: rendono il grafo denso e illeggibile in Gephi senza dire molto sui temi ricorrenti. Il grafo diluito tiene solo gli archi con peso ≥ `MIN_EDGE_WEIGHT` e rimuove i nodi rimasti isolati.

La procedura è la stessa del NYT: un istogramma dei pesi, una tabella di cosa resta a ogni soglia, poi il grafo con la soglia scelta. La soglia è 5, la stessa del NYT, scelta lì a mano sulla distribuzione dei pesi; usarla anche qui permette di confrontare le due reti a parità di regola, anche se la stessa soglia taglia in modo diverso due corpus di dimensioni diverse (§5).

In [ ]:
def prune_graph(graph: nx.Graph, min_weight: int) -> nx.Graph:
    """Copia del grafo con i soli archi di peso >= min_weight; i nodi rimasti isolati sono rimossi."""
    pruned = graph.copy()
    pruned.remove_edges_from([(u, v) for u, v, w in graph.edges(data="weight") if w < min_weight])
    pruned.remove_nodes_from(list(nx.isolates(pruned)))
    return pruned


def threshold_table(graph: nx.Graph, thresholds: list[int]) -> pd.DataFrame:
    """Per ogni soglia: nodi e archi rimasti, componenti connesse, dimensione della componente gigante e quota del peso totale conservata."""
    total_weight = sum(w for _, _, w in graph.edges(data="weight"))
    rows = []
    for t in thresholds:
        pruned = prune_graph(graph, t)
        giant = max((len(c) for c in nx.connected_components(pruned)), default=0)
        rows.append({
            "soglia": t,
            "nodi": pruned.number_of_nodes(),
            "archi": pruned.number_of_edges(),
            "componenti": nx.number_connected_components(pruned),
            "nodi_componente_gigante": giant,
            "peso_conservato_pct": round(sum(w for _, _, w in pruned.edges(data="weight")) / total_weight * 100, 1),
        })
    return pd.DataFrame(rows)


def plot_weight_histogram(graph: nx.Graph, min_weight: int, out_path: Path) -> None:
    """Istogramma dei pesi degli archi in scala logaritmica, con la soglia del grafo diluito; salva il PNG in out_path.

    Assi e bin logaritmici: la distribuzione è a coda lunga (la maggior parte degli archi ha peso 1,
    pochi archi hanno pesi nell'ordine delle centinaia).
    """
    weights = np.array([w for _, _, w in graph.edges(data="weight")])
    # bin logaritmici: pochi archi molto pesanti e una maggioranza con peso 1
    bins = np.unique(np.round(np.logspace(0, np.log10(weights.max() + 1), 30))) - 0.5

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(weights, bins=bins, color="#1f77b4", edgecolor="white", linewidth=0.5)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.axvline(min_weight - 0.5, color="#d62728", linestyle="--", linewidth=1.5, label=f"threshold = {min_weight}")
    ax.set_xlabel("Edge weight (articles in which the two tags co-occur)")
    ax.set_ylabel("Number of edges")
    ax.set_title("Guardian: edge weight distribution in the ICE tag co-occurrence graph")
    ax.legend()
    ax.grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


plot_weight_histogram(graph, MIN_EDGE_WEIGHT, OUT_WEIGHT_HIST_PNG)
shutil.copyfile(OUT_WEIGHT_HIST_PNG, FIG_DIR / "guardian_edge_weight_hist.png")  # copia per la tesi
log.info("Scritto %s", OUT_WEIGHT_HIST_PNG.relative_to(BASE_DIR))
display(Image(filename=str(OUT_WEIGHT_HIST_PNG)))
threshold_table(graph, [1, 2, 3, 4, 5, 8, 10, 15, 20, 30, 50])

L'istogramma (assi logaritmici) mostra una distribuzione a coda lunga: il 63% degli archi ha peso 1, pochi archi superano le centinaia. La linea tratteggiata è la soglia 5.

La tabella mostra cosa resta a ogni soglia. Con soglia 5 restano 172 nodi su 609 (28%) e 911 archi su 7.025 (13%), ma il 73,1% del peso totale: gli archi tolti sono numerosi ma leggeri. Il grafo resta connesso a quasi tutte le soglie (una sola componente, tranne a soglia 4), quindi la diluizione non spezza la rete in isole. La perdita di peso è graduale, senza un punto di rottura evidente: la soglia resta una scelta, motivata dal confronto con il NYT più che dalla forma della curva.

In [ ]:
pruned_graph = prune_graph(graph, MIN_EDGE_WEIGHT)
log.info(
    "Grafo diluito (peso >= %d): %d nodi, %d archi (da %d nodi, %d archi)",
    MIN_EDGE_WEIGHT, pruned_graph.number_of_nodes(), pruned_graph.number_of_edges(),
    graph.number_of_nodes(), graph.number_of_edges(),
)
export_graph(pruned_graph, OUT_PRUNED_GEXF)
log.info("Scritto %s", OUT_PRUNED_GEXF.relative_to(BASE_DIR))

# Controllo: i 15 archi più pesanti devono essere coppie di temi sensate
top_edges = sorted(pruned_graph.edges(data="weight"), key=lambda e: (-e[2], e[0], e[1]))[:15]
pd.DataFrame(
    [(graph.nodes[u]["label"], graph.nodes[v]["label"], w) for u, v, w in top_edges],
    columns=["tag_a", "tag_b", "weight"],
)

Il grafo diluito ha 172 nodi e 911 archi. Gli archi più pesanti collegano i tag generali tra loro: `US news`–`US immigration` (1.311 articoli), `US news`–`Trump administration`, `US news`–`US politics`. Il primo arco con un tag specifico, `US news`–`California` (163), compare solo al quindicesimo posto. Nel grafo pesato la struttura è dominata dai tag generali e di sezione, e per leggere i temi servono pesi normalizzati come quelli usati nel notebook 10.

## 4. Grafi per periodo e grafo dinamico

Per ciascuno dei quattro periodi del progetto (2003-2016, 2017-2020, 2021-2024, 2025-presente) si costruisce un grafo con gli stessi criteri, usando solo gli articoli del periodo. I grafi per periodo **non** sono diluiti: con 90-283 articoli nei periodi più vecchi, una soglia di 5 lascerebbe pochissimi archi.

Il **grafo dinamico** unisce i quattro grafi in un unico file per la Timeline di Gephi: ogni nodo e ogni arco ha degli intervalli di presenza (`spells`, uno per periodo in cui compare) e il conteggio o il peso in ciascun periodo. Permette di vedere in Gephi quali tag e legami compaiono, crescono o scompaiono nel tempo.

Per ogni periodo si salvano nodi, archi, densità e grado pesato medio in `topic_network_period_stats.csv`.

In [ ]:
def get_periods(df: pd.DataFrame) -> list[tuple[str, int, int]]:
    """Confini dei quattro periodi: (etichetta, anno iniziale incluso, anno finale escluso)."""
    max_year = int(df["pub_date"].dt.year.max())
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def filter_by_period(df: pd.DataFrame, start_year: int, end_year: int) -> pd.DataFrame:
    """Articoli pubblicati negli anni [start_year, end_year)."""
    years = df["pub_date"].dt.year
    return df[(years >= start_year) & (years < end_year)]


def graph_stats(graph: nx.Graph) -> dict:
    """Metriche di base per confrontare grafi diversi: nodi, archi, densità e grado pesato medio.

    Il grado pesato medio è 2 × (somma dei pesi) / nodi: quante co-occorrenze tocca in media un nodo.
    """
    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()
    density = nx.density(graph) if n_nodes > 1 else 0.0
    avg_weighted_degree = (
        sum(w for _, _, w in graph.edges(data="weight")) * 2 / n_nodes if n_nodes else 0.0
    )
    return {"nodes": n_nodes, "edges": n_edges, "density": density, "avg_weighted_degree": avg_weighted_degree}


def build_period_graphs(df: pd.DataFrame, periods: list[tuple[str, int, int]]) -> dict:
    """Costruisce un grafo di co-occorrenza per ciascun periodo; restituisce {etichetta: (grafo, inizio, fine, n_articoli)}."""
    period_graphs = {}
    for label, start, end in periods:
        period_df = filter_by_period(df, start, end)
        pgraph = build_cooccurrence_graph(period_df)
        period_graphs[label] = (pgraph, start, end, len(period_df))
        log.info(
            "Periodo %s (%d articoli): %d nodi, %d archi",
            label, len(period_df), pgraph.number_of_nodes(), pgraph.number_of_edges(),
        )
    return period_graphs


def export_period_stats(period_graphs: dict, path: Path) -> None:
    """Scrive in CSV le metriche di graph_stats per ogni periodo, con il numero di articoli."""
    rows = []
    for label, (pgraph, start, end, n_articles) in period_graphs.items():
        rows.append({
            "period": label, "start_year": start, "end_year_excl": end,
            "n_articles": n_articles, **graph_stats(pgraph),
        })
    pd.DataFrame(rows).to_csv(path, index=False)


def build_dynamic_graph(period_graphs: dict) -> nx.Graph:
    """Unisce i grafi per periodo in un grafo dinamico per la Timeline di Gephi.

    Ogni nodo e ogni arco riceve un intervallo (spell) per ciascun periodo in cui compare, e il
    conteggio (nodi) o il peso (archi) in quel periodo; count e weight totali sono la somma sui periodi.
    """
    merged = nx.Graph()
    merged.graph["mode"] = "dynamic"

    for label, (pgraph, start, end, _n_articles) in period_graphs.items():
        start, end = float(start), float(end)
        for node, data in pgraph.nodes(data=True):
            if not merged.has_node(node):
                merged.add_node(node, label=data["label"], count=0, spells=[], count_by_period=[])
            merged.nodes[node]["count"] += data["count"]
            merged.nodes[node]["spells"].append((start, end))
            merged.nodes[node]["count_by_period"].append((data["count"], start, end))

        for u, v, edata in pgraph.edges(data=True):
            if not merged.has_edge(u, v):
                merged.add_edge(u, v, weight=0, spells=[], weight_by_period=[])
            merged[u][v]["weight"] += edata["weight"]
            merged[u][v]["spells"].append((start, end))
            merged[u][v]["weight_by_period"].append((edata["weight"], start, end))

    return merged


# Grafi per periodo, statistiche, un GEXF per periodo e il grafo dinamico
periods = get_periods(df)
period_graphs = build_period_graphs(df, periods)

export_period_stats(period_graphs, OUT_PERIOD_STATS)
log.info("Scritto %s", OUT_PERIOD_STATS.relative_to(BASE_DIR))

for label, (period_graph, *_rest) in period_graphs.items():
    out_path = PROCESSED_DIR / f"ice_topic_network_{label}.gexf"
    export_graph(period_graph, out_path)
    log.info("Scritto %s", out_path.relative_to(BASE_DIR))

dynamic_graph = build_dynamic_graph(period_graphs)
export_graph(dynamic_graph, OUT_DYNAMIC_GEXF)
log.info(
    "Scritto %s (%d nodi, %d archi, mode=dynamic)",
    OUT_DYNAMIC_GEXF.relative_to(BASE_DIR), dynamic_graph.number_of_nodes(), dynamic_graph.number_of_edges(),
)

pd.read_csv(OUT_PERIOD_STATS)

I grafi per periodo hanno 96 / 196 / 100 / 511 nodi e 602 / 1.262 / 588 / 5.885 archi, su 108 / 283 / 90 / 1.257 articoli. Densità e grado pesato medio non sono confrontabili direttamente tra periodi: la densità diminuisce meccanicamente quando i nodi aumentano (0,132 nel 2003-2016, 0,045 nel 2025-presente), e il grado pesato cresce con il numero di articoli (103 nel 2025-presente contro 25-35 nei periodi precedenti). Le differenze riflettono soprattutto la dimensione dei periodi. Per confrontare i temi tra periodi servono misure normalizzate, come le quote di articoli usate nel topic modeling (notebook 07) o la presenza delle comunità per periodo (notebook 10).

Il grafo dinamico contiene tutti i 609 nodi e 7.025 archi del grafo intero, ciascuno con i periodi in cui compare.

## 5. Confronto di struttura NYT vs Guardian

Le metriche sono affiancate per i grafi interi e per quelli diluiti dei due giornali. I grafi NYT sono letti dai GEXF del notebook 02, che va eseguito prima.

Le due tassonomie non sono allineate: gli stessi temi hanno nomi diversi (`illegal immigration` nel NYT, `US immigration` nel Guardian), e il NYT usa più keyword per articolo. Qui si confronta quindi la **forma** delle reti, non i singoli nodi; il confronto dei temi è fatto nel notebook 10, allineando a mano le comunità.

Metriche: nodi, archi, densità, grado pesato medio, componenti connesse e dimensione della componente gigante, peso mediano e massimo degli archi, quota di archi con peso 1 e coefficiente di clustering medio (quanto i vicini di un nodo sono collegati tra loro).

In [ ]:
def comparison_row(name: str, graph: nx.Graph) -> dict:
    """Una riga di metriche di struttura per il grafo dato: quelle di graph_stats più componenti, peso degli archi e clustering medio."""
    giant = max((len(c) for c in nx.connected_components(graph)), default=0)
    weights = np.array([w for _, _, w in graph.edges(data="weight")])
    return {
        "grafo": name,
        **graph_stats(graph),
        "componenti": nx.number_connected_components(graph),
        "nodi_componente_gigante": giant,
        "peso_mediano": float(np.median(weights)),
        "peso_max": int(weights.max()),
        "archi_peso_1_pct": round(float((weights == 1).mean() * 100), 1),
        "clustering_medio": round(nx.average_clustering(graph), 3),
    }


# Grafi interi e diluiti dei due giornali; quelli NYT letti dal notebook 02
comparison = pd.DataFrame([
    comparison_row("NYT intero", nx.read_gexf(IN_NYT_GEXF)),
    comparison_row("Guardian intero", graph),
    comparison_row(f"NYT diluito (>= {MIN_EDGE_WEIGHT})", nx.read_gexf(IN_NYT_PRUNED_GEXF)),
    comparison_row(f"Guardian diluito (>= {MIN_EDGE_WEIGHT})", pruned_graph),
])
comparison.to_csv(OUT_COMPARISON, index=False)
log.info("Scritto %s", OUT_COMPARISON.relative_to(BASE_DIR))
comparison

Le due reti hanno forme diverse per ragioni di tassonomia più che di contenuto.

- **Dimensione.** Il NYT ha un vocabolario quattro volte più ampio (2.441 keyword contro 609 tag) e una rete molto più rada (densità 0,012 contro 0,038).
- **Concentrazione del peso.** Il 75,8% degli archi NYT ha peso 1, contro il 63,0% del Guardian. L'arco più pesante del Guardian è la coppia di tag generali `US news`–`US immigration` (1.311 articoli); quello del NYT è `united states politics and government`–`trump, donald j` (715).
- **Clustering.** Alto in entrambe (0,83 e 0,82). È un effetto della costruzione: ogni articolo collega tutti i suoi tag tra loro, quindi i vicini di un nodo tendono a essere collegati. Non va letto come una proprietà dei temi.
- **Grafi diluiti.** La stessa soglia 5 lascia il 15% dei nodi NYT (369) e il 28% dei nodi Guardian (172). Il grafo NYT diluito ha 2 componenti, quello Guardian una sola.

La figura successiva mostra come cambia la diluizione al variare della soglia per i due giornali.

In [ ]:
# Quota di nodi e di peso conservata a ogni soglia, per i grafi interi dei due giornali
thresholds = [1, 2, 3, 4, 5, 8, 10, 15, 20, 30, 50]
curves = {}
for name, g in [("NYT", nx.read_gexf(IN_NYT_GEXF)), ("Guardian", graph)]:
    t = threshold_table(g, thresholds)
    t["nodi_pct"] = 100 * t["nodi"] / g.number_of_nodes()
    curves[name] = t

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
colors = {"NYT": "#555555", "Guardian": "#2a78d6"}
for ax, col, title in zip(axes, ["nodi_pct", "peso_conservato_pct"], ["Nodes kept (%)", "Total edge weight kept (%)"]):
    for name, t in curves.items():
        ax.plot(t["soglia"], t[col], marker="o", color=colors[name], label=name)
    ax.axvline(MIN_EDGE_WEIGHT, color="#d62728", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks(thresholds, [str(t) for t in thresholds])
    ax.set_ylim(0, 102)
    ax.set_title(title)
    ax.set_xlabel("minimum edge weight")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
fig.suptitle("Pruning the co-occurrence graphs: NYT vs Guardian (dashed line: threshold used, 5)", fontsize=11)
plt.tight_layout()
fig.savefig(FIG_DIR / "topic_network_pruning_nyt_vs_guardian.png", dpi=200, bbox_inches="tight")
plt.show()

A sinistra la quota di nodi che resta nel grafo diluito, a destra la quota del peso totale, al variare della soglia (asse logaritmico). Il Guardian conserva più nodi e molto più peso a ogni soglia: con soglia 5 tiene il 28% dei nodi e il 73% del peso, il NYT il 15% e il 49%. Nel Guardian il peso è concentrato su pochi archi molto pesanti, in buona parte tra tag generali e di sezione (§2), mentre nel NYT è distribuito su molte keyword specifiche. La stessa soglia non produce quindi grafi diluiti equivalenti, e il confronto dei grafi diluiti va letto tenendo conto di questa differenza. La figura è salvata in `docs/figures/topic_network_pruning_nyt_vs_guardian.png`.

## 6. Lettura dei risultati e limiti

Run del 2026-09-30, rieseguito il 2026-10-05 (pulizia delle keyword NYT nel notebook 02) e il 2026-10-08 (testo e figure; grafi identici).

**Risultati.**

- La rete del Guardian ha 609 nodi e 7.025 archi (diluita: 172 e 911), costruita con lo stesso codice e la stessa soglia del NYT.
- È più piccola e più densa di quella del NYT (2.441 nodi, 36.654 archi) perché il Guardian usa meno tag diversi e meno tag per articolo (mediana 6 contro 8).
- Il peso è concentrato su tag generali e di sezione: `US news` è in quasi tutti gli articoli e il 38,8% del peso degli archi tocca un tag di sezione. A parità di soglia il Guardian conserva il 73% del peso, il NYT il 49%.
- I grafi per periodo crescono con il numero di articoli; densità e grado pesato riflettono soprattutto la dimensione dei periodi.

**Limiti per il confronto.**

- **Tassonomie non allineate.** Si confronta la forma delle reti; un confronto nodo per nodo richiede una corrispondenza fatta a mano (notebook 10, §5).
- **Tag di sezione.** `US news`, `World news` e altri 25 tag di sezione fanno da hub. Lasciati per non applicare al Guardian un filtro che il NYT non ha; nelle misure di centralità e nella lettura in Gephi vanno tenuti presenti o nascosti.
- **Tag ICE.** Nel NYT la keyword esclusa è in tutti gli articoli; nel Guardian il tag escluso è in 964 articoli su 1.738, quasi tutti dal 2025.
- **Rank.** La posizione del tag è un sostituto del `rank` del NYT, non la stessa misura.
- **Stessa soglia, effetti diversi.** La soglia 5 lascia il 15% dei nodi NYT e il 28% di quelli Guardian.
- **Corpus selezionati in modo diverso.** NYT per keyword di redazione, Guardian per regola + classificatore di rilevanza (notebook 05).